# Setup for STOCKS and ETF etc.

# Analyze Visul

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def Analyze_price(tickers, df_data_raw):
    """
    Analyze stock data: plot Adjusted Close, cumulative returns, and cumulative log returns.
    """
    # --- Plot Adjusted Close Price ---
    df_adj_close = df_data_raw['Adj Close']
    df_adj_close[tickers].plot(figsize=(12,6), linewidth=2)
    plt.title('Adjusted Close Price Over Time', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Adjusted Close Price', fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot Cumulative Percentage Differences ---
    df_percentage_diff = df_adj_close.pct_change().cumsum()
    df_percentage_diff.iloc[0] = 0  # start at 0
    df_percentage_diff[tickers].plot(figsize=(12,6), linewidth=1.5)
    plt.title('Cumulative Percentage Difference in Adjusted Close Price', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Cumulative % Difference', fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Plot Cumulative Sum of Log Differences ---
    df_log_diff = np.log(df_adj_close / df_adj_close.shift(1))
    df_log_diff_cumsum = df_log_diff.cumsum()
    df_log_diff_cumsum.iloc[0] = 0  # start at 0
    df_log_diff_cumsum[tickers].plot(figsize=(12,6), linewidth=1.5)
    plt.title('Cumulative Sum of Log Differences in Adjusted Close Price', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Cumulative Log Difference', fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def Analyze_price_2(tickers, df_data_raw, window=30):
    """
    Analyze stock data: plot Adjusted Close, cumulative returns,
    cumulative log returns, rolling MA, rolling volatility,
    and rolling MA/vol ratio.

    Only plots data for the tickers provided in the 'tickers' list.

    Args:
        tickers (list): List of stock tickers
        df_data_raw (pd.DataFrame): Downloaded stock data
        window (int): Rolling window size in days (default=30)
    """
    # Filter only the tickers we want
    df_adj_close = df_data_raw["Adj Close"][tickers]

    # --- Cumulative Returns ---
    df_cum_returns = df_adj_close.pct_change().cumsum()

    # --- Rolling Mean of Cumulative Returns ---
    rolling_ma = df_cum_returns.rolling(window=window).mean()
    rolling_ma.plot(figsize=(12,6), linewidth=2)
    plt.title(f"Rolling {window}-Day Mean of Cumulative Returns", fontsize=14)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Rolling Mean", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()


    # --- Rolling Std of Daily Returns ---
    daily_returns = df_adj_close.pct_change()
    rolling_std = daily_returns.rolling(window=window).std()
    rolling_std.plot(figsize=(12,6), linewidth=2)
    plt.title(f"Rolling {window}-Day Standard Deviation of Daily Returns", fontsize=14)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Rolling Std", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Rolling Mean / Rolling Std Ratio ---
    ratio = rolling_ma / rolling_std
    ratio.plot(figsize=(12,6), linewidth=2)
    plt.title(f"Rolling {window}-Day Mean of Cumulative Returns / Std of Returns", fontsize=14)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Ratio", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Drawdowns ---
    rolling_max = df_adj_close.cummax()
    drawdown = (df_adj_close - rolling_max) / rolling_max
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(drawdown[ticker], label=f"{ticker} Drawdown")
    plt.title("Drawdowns (Peak-to-Trough Losses)", fontsize=14)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Drawdown", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

def Analyze_price_return_dynamics(tickers, df_data_raw, windows=[30, 60, 90]):
    """
    Analyze stock price and return dynamics:
    - Cumulative returns (linear & log)
    - Rolling returns (annualized)
    - Rolling volatility
    - Skewness & kurtosis of returns

    Args:
        tickers (list): List of stock tickers
        df_data_raw (pd.DataFrame): Downloaded stock data
        windows (list): Rolling window sizes in days for rolling metrics
    """
    df_adj_close = df_data_raw["Adj Close"]

    # --- Daily Returns ---
    daily_returns = df_adj_close.pct_change()


    # --- Rolling Returns & Volatility ---
    for window in windows:
        rolling_ret = daily_returns.rolling(window=window).mean() * 252  # annualized
        rolling_vol = daily_returns.rolling(window=window).std() * np.sqrt(252)  # annualized

        plt.figure(figsize=(12,6))
        for ticker in tickers:
            plt.plot(rolling_ret[ticker], label=f"{ticker} {window}-day Rolling Return")
        plt.title(f"Rolling {window}-Day Annualized Returns", fontsize=14)
        plt.xlabel("Date", fontsize=12)
        plt.ylabel("Annualized Return", fontsize=12)
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12,6))
        for ticker in tickers:
            plt.plot(rolling_vol[ticker], label=f"{ticker} {window}-day Rolling Volatility")
        plt.title(f"Rolling {window}-Day Annualized Volatility", fontsize=14)
        plt.xlabel("Date", fontsize=12)
        plt.ylabel("Annualized Volatility", fontsize=12)
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- Skewness & Kurtosis ---
    skewness = daily_returns.apply(lambda x: skew(x.dropna()))
    kurt = daily_returns.apply(lambda x: kurtosis(x.dropna()))

    skew_kurt_df = pd.DataFrame({"Skewness": skewness, "Kurtosis": kurt})
    print("\nSkewness & Kurtosis of Daily Returns:")
    print(skew_kurt_df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.stattools import coint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import scipy.stats as stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

import pandas as pd
import talib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np # Import numpy






import statsmodels as sm

def Analyze_asset_relationships(tickers, df_data_raw, market_index="^GSPC", rolling_window=60):
    """
    Analyze relationships between assets:
    - Correlation matrix & heatmap
    - Rolling correlations
    - Beta vs market
    - Cointegration tests

    Args:
        tickers (list): List of stock tickers
        df_data_raw (pd.DataFrame): Downloaded stock data (Close)
        market_index (str): Market index ticker for beta calculation (default S&P500 ^GSPC)
        rolling_window (int): Window for rolling correlation
    """
    df_adj_close = df_data_raw["Close"]

    # --- Daily Returns ---
    daily_returns = df_adj_close.pct_change().dropna()

    # --- Correlation Matrix ---
    corr_matrix = daily_returns.corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
    plt.title("Correlation Matrix Between Assets", fontsize=14)
    plt.show()

    # --- Rolling Correlations ---
    plt.figure(figsize=(12,6))
    for i in range(len(tickers)):
        for j in range(i+1, len(tickers)):
            rolling_corr = daily_returns[tickers[i]].rolling(rolling_window).corr(daily_returns[tickers[j]])
            plt.plot(rolling_corr, label=f"{tickers[i]} vs {tickers[j]}")
    plt.title(f"Rolling {rolling_window}-Day Correlation Between Assets", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Correlation")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()



    # --- Cointegration Tests ---
    coint_results = {}
    for i in range(len(tickers)):
        for j in range(i+1, len(tickers)):
            score, pvalue, _ = coint(df_adj_close[tickers[i]], df_adj_close[tickers[j]])
            coint_results[f"{tickers[i]} & {tickers[j]}"] = pvalue
    coint_df = pd.DataFrame.from_dict(coint_results, orient="index", columns=["p-value"])
    print("\nCointegration Test p-values (pairs with p < 0.05 are significant):")
    print(coint_df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import scipy.stats as stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

import pandas as pd
import talib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np # Import numpy





def Analyze_volume_liquidity_and_stylized_facts(tickers, df_data_raw, rolling_window=30):
    """
    Analyze volume, liquidity, and stylized facts of returns:
    - Volume trends
    - Price-volume relationship
    - Volatility vs volume
    - Autocorrelation (ACF/PACF) of returns
    - Volatility clustering visualization
    - QQ-plots for return distribution

    Args:
        tickers (list): List of stock tickers
        df_data_raw (pd.DataFrame): Downloaded stock data (Adj Close & Volume)
        rolling_window (int): Window for rolling metrics
    """
    df_adj_close = df_data_raw["Adj Close"]
    df_volume = df_data_raw["Volume"]

    daily_returns = df_adj_close.pct_change().dropna()

    # --- Volume Trends ---
    rolling_vol = df_volume.rolling(window=rolling_window).mean()
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(df_volume[ticker], alpha=0.5, label=f"{ticker} Daily Volume")
        plt.plot(rolling_vol[ticker], linewidth=2, label=f"{ticker} {rolling_window}-day Rolling Volume")
    plt.title("Volume Trends & Rolling Average", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Price-Volume Relationship ---
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.scatter(df_volume[ticker], df_adj_close[ticker], alpha=0.5, label=ticker)
    plt.title("Price vs Volume", fontsize=14)
    plt.xlabel("Volume")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


    # --- Volatility Clustering (Squared Returns) ---
    squared_returns = daily_returns**2
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(squared_returns[ticker], label=f"{ticker} Squared Returns")
    plt.title("Volatility Clustering (Squared Returns)", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Squared Returns")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

def Analyze_volume_liquidity_and_stylized_facts(tickers, df_data_raw, rolling_window=30):
    """
    Comprehensive analysis of volume, liquidity, and stylized facts of returns:
    - Volume trends & rolling averages
    - Volume spikes / extremes
    - Price-volume relationship
    - Volatility vs volume
    - Volume-weighted average price (VWAP)
    - Price distribution weighted by volume
    - On-Balance Volume (OBV)
    - Volatility clustering visualization (squared returns)
    - Log returns vs volume

    Args:
        tickers (list): List of stock tickers
        df_data_raw (pd.DataFrame): Downloaded stock data (Adj Close & Volume)
        rolling_window (int): Window for rolling metrics
    """
    df_adj_close = df_data_raw["Adj Close"]
    df_volume = df_data_raw["Volume"]
    # Change to log returns
    daily_returns = np.log(df_adj_close / df_adj_close.shift(1)).dropna()

    # --- 1. Volume Trends ---
    rolling_vol = df_volume.rolling(window=rolling_window).mean()
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(df_volume[ticker], alpha=0.5, label=f"{ticker} Daily Volume")
        plt.plot(rolling_vol[ticker], linewidth=2, label=f"{ticker} {rolling_window}-day Rolling Volume")
    plt.title("Volume Trends & Rolling Average", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- 2. Volume Spikes / Extremes ---
    volume_spike = df_volume / rolling_vol
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(volume_spike[ticker], label=f"{ticker} Volume Spike Ratio")
    plt.axhline(2, color='r', linestyle='--', label='2x Rolling Avg')
    plt.title("Volume Spikes Relative to Rolling Mean")
    plt.legend()
    plt.show()

    # --- 3. Price-Volume Relationship ---
    plt.figure(figsize=(10,6))
    for ticker in tickers:
        df = pd.concat([df_volume[ticker], df_adj_close[ticker]], axis=1).dropna()
        # Access the price column using the ticker name
        plt.scatter(df[ticker], df[ticker], alpha=0.5, label=ticker)
    plt.title("Price vs Volume")
    plt.xlabel("Volume")
    plt.ylabel("Price")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- 4. Volatility vs Volume ---
    rolling_volatility = daily_returns.rolling(window=rolling_window).std()
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        # Ensure both series have the same index before plotting
        volume_aligned, volatility_aligned = df_volume[ticker].align(rolling_volatility[ticker], join='inner')
        plt.scatter(volume_aligned, volatility_aligned, alpha=0.5, label=ticker)
    plt.title("Volatility vs Volume")
    plt.xlabel("Volume")
    plt.ylabel(f"{rolling_window}-day Rolling Volatility (Log Returns)") # Updated title
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- 5. Volume-Weighted Average Price (VWAP) ---
    vwap = (df_adj_close * df_volume).cumsum() / df_volume.cumsum()
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(vwap[ticker], label=f"{ticker} VWAP")
    plt.title("Volume-Weighted Average Price (VWAP)")
    plt.legend()
    plt.show()

    # --- 6. Price Distribution Weighted by Volume ---
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        # Plot distribution of log returns weighted by volume
        plt.hist(daily_returns[ticker].dropna(), weights=df_volume[ticker].loc[daily_returns[ticker].dropna().index], bins=50, alpha=0.5, label=ticker)
    plt.title("Log Return Distribution Weighted by Volume") # Updated title
    plt.xlabel("Log Return") # Updated label
    plt.ylabel("Weighted Volume")
    plt.legend()
    plt.show()

    # --- 7. On-Balance Volume (OBV) ---
    obv = pd.DataFrame(index=df_adj_close.index)
    for ticker in tickers:
        obv[ticker] = np.where(df_adj_close[ticker].diff() > 0, df_volume[ticker],
                                np.where(df_adj_close[ticker].diff() < 0, -df_volume[ticker], 0)).cumsum()
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(obv[ticker], label=f"{ticker} OBV")
    plt.title("On-Balance Volume (OBV)")
    plt.legend()
    plt.show()

    # --- 8. Volatility Clustering (Squared Returns) ---
    squared_returns = daily_returns**2
    plt.figure(figsize=(12,6))
    for ticker in tickers:
        plt.plot(squared_returns[ticker], label=f"{ticker} Squared Log Returns") # Updated title
    plt.title("Volatility Clustering (Squared Log Returns)") # Updated title
    plt.xlabel("Date")
    plt.ylabel("Squared Log Returns") # Updated label
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()


    # --- 11. Log Returns vs Volume ---
    plt.figure(figsize=(10,6))
    for ticker in tickers:
        # Ensure both series have the same index before plotting
        volume_aligned, returns_aligned = df_volume[ticker].align(daily_returns[ticker], join='inner')
        plt.scatter(volume_aligned, returns_aligned, alpha=0.5, label=ticker)
    plt.title("Log Returns vs Volume")
    plt.xlabel("Volume")
    plt.ylabel("Log Returns")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Feature Engineer

In [ ]:
class FeatureEngineerTA:
    def __init__(self, df_data_raw, tickers):
        """
        Initialize with multi-level OHLC DataFrame from yfinance and tickers to process.

        Args:
            df_data_raw (pd.DataFrame): Multi-index columns DataFrame (OHLC + Volume)
            tickers (list): List of tickers to process
        """
        self.tickers = tickers
        self.df_data_raw = df_data_raw.copy()
        # MultiIndex: level 0 = ticker, level 1 = feature
        self.features = pd.DataFrame(index=df_data_raw.index)


    # -------------------- Get Features --------------------
    def get_features(self, sort_columns=True):
        """Return features with MultiIndex columns (Ticker, Feature)."""
        if self.features.empty:
            return self.features.copy()

        if not isinstance(self.features.columns, pd.MultiIndex):
            self.features.columns = pd.MultiIndex.from_tuples(self.features.columns, names=["Ticker", "Feature"])
        else:
            self.features.columns = self.features.columns.set_names(["Ticker", "Feature"])

        out = self.features
        return out.sort_index(axis=1).copy() if sort_columns else out.copy()


    # -------------------- Get Features --------------------
    def add_lagged_features_returns(self,
                                    log_returns_lags=[1,2,3],
                                    rolling_windows=[3,7,14],
                                    cumdiff_windows=[3,5,7,9,15,40]):
        df_close = self.df_data_raw["Adj Close"]

        for ticker in self.tickers:
            # --- Log returns ---
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))

            # Lagged log returns
            for lag in log_returns_lags:
                self.features[(ticker, f"logret_lag{lag}")] = log_ret.shift(lag)

            # Rolling statistics
            for window in rolling_windows:
                self.features[(ticker, f"ret_mean_{window}")] = log_ret.rolling(window).mean()
                self.features[(ticker, f"ret_std_{window}")] = log_ret.rolling(window).std()
                self.features[(ticker, f"ret_skew_{window}")] = log_ret.rolling(window).skew()
                #self.features[(ticker, f"ret_kurt_{window}")] = log_ret.rolling(window).kurt()

            # Streaks
            self.features[(ticker, f"pos_streak")] = (log_ret > 0).astype(int).groupby((log_ret <= 0).cumsum()).cumsum()
            self.features[(ticker, f"neg_streak")] = (log_ret < 0).astype(int).groupby((log_ret >= 0).cumsum()).cumsum()

         #   # --- Cumulated price change over given windows ---
         #   for window in cumdiff_windows:
         #       # relative change compared to 'window' days ago
         #       self.features[(ticker, f"cumdiff_{window}")] = df_close[ticker] / df_close[ticker].shift(window) - 1
         #       # or in log terms (compounded change)
         #       # self.features[(ticker, f"log_cumdiff_{window}")] = np.log(df_close[ticker] / df_close[ticker].shift(window))

        return self


    # -------------------- Get Features --------------------
    def add_lagged_features_volume(self, returns_lags=[1,2,3], log_returns_lags=[1,2,3], rolling_windows=[3,7,14]):
      df_close = self.df_data_raw["Volume"]

      for ticker in self.tickers:
          # --- Log returns ---
          log_ret = (df_close[ticker].pct_change() + 1).apply(np.log)

          # Lagged log returns
          for lag in log_returns_lags:
              self.features[(ticker, f"logret_lag{lag}")] = log_ret.shift(lag)


          # Rolling statistics
          for window in rolling_windows:
              self.features[(ticker, f"ret_mean_{window}")] = log_ret.rolling(window).mean()
              self.features[(ticker, f"ret_std_{window}")] = log_ret.rolling(window).std()
              self.features[(ticker, f"ret_skew_{window}")] = log_ret.rolling(window).skew()
              #self.features[(ticker, f"ret_kurt_{window}")] = log_ret.rolling(window).kurt()

          # Streaks
          self.features[(ticker, f"pos_streak")] = (log_ret > 0).astype(int).groupby((log_ret <= 0).cumsum()).cumsum()
          self.features[(ticker, f"neg_streak")] = (log_ret < 0).astype(int).groupby((log_ret >= 0).cumsum()).cumsum()

      return self

    # -------------------- Get Features --------------------
    def add_rolling_percentile(self, window=20, ma_window=10, price_type="Adj Close"):
        """
        Compute rolling percentile of the current price compared to the past `window` days.
        Optionally compute moving average of the percentile.

        Args:
            window (int): Lookback window for percentile calculation.
            ma_window (int or None): If set, compute rolling mean of the percentile over `ma_window` days.
            price_type (str): Column of price to use ("Adj Close" by default).
        """
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]

            # Rolling apply: percentile of current value in rolling window
            def percentile_in_window(x):
                ranks = np.argsort(np.argsort(x))  # 0..N-1
                return ranks[-1] / (len(x) - 1)

            percentile_series = series.rolling(window).apply(percentile_in_window, raw=True)
            self.features[(ticker, f"{price_type}_pct_{window}")] = percentile_series

            # Optional: moving average of percentile
            if ma_window is not None:
                self.features[(ticker, f"{price_type}_pctMA_{window}_{ma_window}")] = percentile_series.rolling(ma_window).mean()

        return self

    # -------------------- EMA --------------------
    def add_EMA(self, n=[10, 20, 50], price_type='Adj Close'):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for period in n:  # iterate over list
                col = (ticker, f"EMA_{period}")
                self.features[col] = talib.EMA(series, timeperiod=period)
        return self

    # -------------------- SMA --------------------
    def add_SMA(self, n=[10, 20, 50], price_type='Adj Close'):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for period in n:
                col = (ticker, f"SMA_{period}")
                self.features[col] = talib.SMA(series, timeperiod=period)
        return self

    # -------------------- EMA --------------------
    def add_EMA_log_dist(self, n=[10, 20, 50], price_type='Adj Close'):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for period in n:
                ema = talib.EMA(series, timeperiod=period)
                col = (ticker, f"EMA_log_dist_{period}")
                self.features[col] = np.log(series / ema)
        return self

    # -------------------- SMA --------------------
    def add_SMA_log_dist(self, n=[10, 20, 50], price_type='Adj Close'):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for period in n:
                sma = talib.SMA(series, timeperiod=period)
                col = (ticker, f"SMA_log_dist_{period}")
                self.features[col] = np.log(series / sma)
        return self
    # -------------------- RSI --------------------
    def add_RSI(self, n=[14], price_type='Adj Close'):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for period in n:  # handle list of periods
                col = (ticker, f"RSI_{period}")
                self.features[col] = talib.RSI(series, timeperiod=period)
        return self

    # -------------------- MACD --------------------
    def add_MACD(self, params=[(12, 26, 9)], price_type='Adj Close'):
        """
        Add MACD indicators for multiple parameter sets.

        Args:
            params (list of tuples): Each tuple is (fast, slow, signal)
            price_type (str): Price column to use ('Adj Close' by default)
        """
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for fast, slow, signal in params:  # iterate over list of tuples
                macd, macdsignal, macdhist = talib.MACD(
                    series, fastperiod=fast, slowperiod=slow, signalperiod=signal
                )
                self.features[(ticker, f"MACD_{fast}_{slow}_{signal}")] = macd
                self.features[(ticker, f"MACD_signal_{fast}_{slow}_{signal}")] = macdsignal
                self.features[(ticker, f"MACD_hist_{fast}_{slow}_{signal}")] = macdhist
        return self

    def add_MACD_log_dist(self, params=[(12, 26, 9)], price_type='Adj Close'):
        """
        Add MACD indicators as log-distance from price.

        Args:
            params (list of tuples): Each tuple is (fast, slow, signal)
            price_type (str): Price column to use ('Adj Close' by default)
        """
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            for fast, slow, signal in params:
                macd, macdsignal, macdhist = talib.MACD(
                    series, fastperiod=fast, slowperiod=slow, signalperiod=signal
                )

                # log-distances
                self.features[(ticker, f"MACD_log_dist_{fast}_{slow}_{signal}")] = np.log(series/macd)
                self.features[(ticker, f"MACD_signal_log_dist_{fast}_{slow}_{signal}")] = np.log(series/macdsignal)

                # keep hist as-is (hist = macd - macdsignal)
                self.features[(ticker, f"MACD_hist_{fast}_{slow}_{signal}")] = macdhist

        return self

    # -------------------- Volume Spikes --------------------
    def add_volume_spikes(self, window=24):
        """
        Add volume spike ratio = Volume / rolling mean(Volume)

        Args:
            window (int): Rolling window size in periods (default 24 for 1 day if 1H data)
        """
        for ticker in self.tickers:
            series = self.df_data_raw["Volume"][ticker]
            rolling_vol = series.rolling(window=window, min_periods=1).mean()
            volume_spike = series / rolling_vol
            self.features[(ticker, f"VolumeSpike_{window}")] = volume_spike
        return self


    # -------------------- Volatility --------------------
    def add_ATR(self, n=14):
        # Average True Range
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            # ATR takes a single timeperiod
            self.features[(ticker, f"ATR_{n}")] = talib.ATR(high, low, close, timeperiod=n)
        return self

    def add_BBANDS(self, n=20, nbdevup=2, nbdevdn=2, matype=0, price_type="Adj Close"):
        # Bollinger Bands
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            # BBANDS takes a single timeperiod
            upper, middle, lower = talib.BBANDS(
                series, timeperiod=n, nbdevup=nbdevup, nbdevdn=nbdevdn, matype=matype
            )
            self.features[(ticker, f"BBANDS_upper_{n}")] = np.log(series/upper)
            self.features[(ticker, f"BBANDS_middle_{n}")] = np.log(series/middle)
            self.features[(ticker, f"BBANDS_lower_{n}")] = np.log(series/lower)
        return self

    def add_NATR(self, n=14):
        # Normalized Average True Range
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            # NATR takes a single timeperiod
            self.features[(ticker, f"NATR_{n}")] = talib.NATR(high, low, close, timeperiod=n)
        return self

    # -------------------- Volume-based --------------------
    def add_OBV(self, price_type="Adj Close"):
        # On Balance Volume
        for ticker in self.tickers:
            price, volume = self.df_data_raw[price_type][ticker], self.df_data_raw["Volume"][ticker]
            self.features[(ticker, "OBV")] = talib.OBV(price, volume)
        return self

    def add_AD(self):
        # Accumulation/Distribution Line
        for ticker in self.tickers:
            high, low, close, volume = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
                self.df_data_raw["Volume"][ticker],
            )
            self.features[(ticker, "AD")] = talib.AD(high, low, close, volume)
        return self

    def add_ADOSC(self, fast=3, slow=10):
        # Accumulation/Distribution Oscillator
        for ticker in self.tickers:
            high, low, close, volume = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
                self.df_data_raw["Volume"][ticker],
            )
            # ADOSC takes single fast and slow periods
            self.features[(ticker, f"ADOSC_{fast}_{slow}")] = talib.ADOSC(high, low, close, volume, fastperiod=fast, slowperiod=slow)
        return self

    def add_MFI(self, n=14):
        # Money Flow Index
        for ticker in self.tickers:
            high, low, close, volume = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
                self.df_data_raw["Volume"][ticker],
            )
            # MFI takes a single timeperiod
            self.features[(ticker, f"MFI_{n}")] = talib.MFI(high, low, close, volume, timeperiod=n)
        return self

    # -------------------- Oscillators --------------------
    def add_STOCH(self, params=[(14, 3, 3)]):
        """
        Add Stochastic Oscillator indicators for multiple parameter sets.

        Args:
            params (list of tuples): Each tuple is (fastk, slowk, slowd)
        """
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            for fastk, slowk, slowd in params:  # iterate over list of tuples
                slowk_series, slowd_series = talib.STOCH(
                    high, low, close,
                    fastk_period=fastk, slowk_period=slowk, slowk_matype=0,
                    slowd_period=slowd, slowd_matype=0
                )
                self.features[(ticker, f"STOCH_slowk_{fastk}_{slowk}_{slowd}")] = slowk_series
                self.features[(ticker, f"STOCH_slowd_{fastk}_{slowk}_{slowd}")] = slowd_series
        return self


    def add_STOCHF(self, fastk=14, fastd=3):
        # Fast Stochastic Oscillator
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            # STOCHF takes single fastk and fastd periods
            fastk_series, fastd_series = talib.STOCHF(
                high, low, close, fastk_period=fastk,
                fastd_period=fastd, fastd_matype=0
            )
            self.features[(ticker, f"STOCHF_fastk_{fastk}_{fastd}")] = fastk_series
            self.features[(ticker, f"STOCHF_fastd_{fastk}_{fastd}")] = fastd_series
        return self

    def add_ULTOSC(self, t1=7, t2=14, t3=28):
        # Ultimate Oscillator
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            # ULTOSC takes single timeperiod1, timeperiod2, and timeperiod3
            self.features[(ticker, f"ULTOSC_{t1}_{t2}_{t3}")] = talib.ULTOSC(high, low, close, timeperiod1=t1, timeperiod2=t2, timeperiod3=t3)
        return self

    def add_WILLR(self, n=14):
        # Williams' %R
        for ticker in self.tickers:
            high, low, close = (
                self.df_data_raw["High"][ticker],
                self.df_data_raw["Low"][ticker],
                self.df_data_raw["Adj Close"][ticker],
            )
            # WILLR takes a single timeperiod
            self.features[(ticker, f"WILLR_{n}")] = talib.WILLR(high, low, close, timeperiod=n)
        return self





    # ... tvoje ostatní metody nahoře ...

    # -------------------- Autocorrelation --------------------
    def add_autocorr(self, lags=[1, 5, 10], price_type="Adj Close"):
        """
        Adds autocorrelation features for log returns.
        """
        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
            for lag in lags:
                self.features[(ticker, f"autocorr_{lag}")] = (
                    log_ret.rolling(lag * 5).apply(lambda x: pd.Series(x).autocorr(lag))
                )
        return self

    # -------------------- Z-score of returns --------------------
    def add_zscore_returns(self, windows=[20, 60, 120], price_type="Adj Close"):
        """
        Adds z-score of log returns over rolling windows.
        """
        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
            for window in windows:
                mean = log_ret.rolling(window).mean()
                std = log_ret.rolling(window).std()
                self.features[(ticker, f"zscore_ret_{window}")] = (log_ret - mean) / std
        return self

    # -------------------- Rolling Entropy --------------------
    def add_entropy(self, windows=[20, 60, 120], price_type="Adj Close", bins=10):
        """
        Adds Shannon entropy of return distributions over rolling windows.
        """
        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
            for window in windows:
                self.features[(ticker, f"entropy_{window}")] = (
                    log_ret.rolling(window)
                    .apply(lambda x: scipy_entropy(np.histogram(x, bins=bins, density=True)[0] + 1e-9), raw=False)
                )
        return self

    # -------------------- Hurst Exponent --------------------
    def add_Hurst(self, windows=[100], price_type="Adj Close"):
        """
        Adds rolling Hurst exponent of log returns.
        """
        def hurst_exponent(ts):
            if len(ts) < 20:  # avoid tiny samples
                return np.nan
            lags = range(2, min(100, len(ts)//2))
            tau = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
            poly = np.polyfit(np.log(lags), np.log(tau), 1)
            return poly[0] * 2.0

        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
            for window in windows:
                self.features[(ticker, f"hurst_{window}")] = (
                    log_ret.rolling(window).apply(hurst_exponent, raw=False)
                )
        return self

    # -------------------- Fractal Dimension --------------------
    def add_fractal_dim(self, windows=[100], price_type="Adj Close"):
        """
        Adds rolling fractal dimension using Katz method approximation.
        """
        def katz_fd(ts):
            if len(ts) < 20:
                return np.nan
            L = np.sum(np.abs(np.diff(ts)))  # total path length
            d = np.max(np.abs(ts - ts[0]))  # max distance from first point
            n = len(ts)
            if d == 0 or L == 0:
                return np.nan
            return np.log10(n) / (np.log10(d / L) + np.log10(n))

        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            series = df_close[ticker]
            for window in windows:
                self.features[(ticker, f"fractal_dim_{window}")] = (
                    series.rolling(window).apply(katz_fd, raw=False)
                )
        return self

######################

# ---  My

    def add_ma_std_ratio(self, windows=[10,20,50], price_type="Adj Close"):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            log_ret = np.log(series / series.shift(1))  # use log returns
            for window in windows:
                rolling_ma = log_ret.rolling(window).mean()
                rolling_std = log_ret.rolling(window).std()
                self.features[(ticker, f"ma_std_ratio_{window}")] = rolling_ma / rolling_std
        return self

    def add_ema_atr_ratio(self, ema_windows=[10,20,50], atr_window=14, price_type="Adj Close"):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            high = self.df_data_raw["High"][ticker]
            low = self.df_data_raw["Low"][ticker]
            close = series
            for period in ema_windows:
                ema = talib.EMA(series, timeperiod=period)
                atr = talib.ATR(high, low, close, timeperiod=atr_window)
                self.features[(ticker, f"ema_atr_ratio_{period}_{atr_window}")] = ema / atr
        return self


    def add_trend_to_risk(self, windows=[10,20,50], price_type="Adj Close"):
            df_close = self.df_data_raw[price_type]
            for ticker in self.tickers:
                log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
                for window in windows:
                    cum_ret = log_ret.rolling(window).sum()
                    rolling_vol = log_ret.rolling(window).std()
                    self.features[(ticker, f"trend_to_risk_{window}")] = cum_ret / rolling_vol
            return self

    def add_rsi_vol_ratio(self, rsi_window=14, vol_window=14, price_type="Adj Close"):
        df_close = self.df_data_raw[price_type]
        for ticker in self.tickers:
            rsi = talib.RSI(df_close[ticker], timeperiod=rsi_window)
            log_ret = np.log(df_close[ticker] / df_close[ticker].shift(1))
            rolling_vol = log_ret.rolling(vol_window).std()
            self.features[(ticker, f"rsi_vol_ratio_{rsi_window}_{vol_window}")] = rsi / rolling_vol
        return self


    def add_bbands_std_ratio(self, n=20, nbdevup=2, nbdevdn=2, matype=0, price_type="Adj Close"):
        for ticker in self.tickers:
            series = self.df_data_raw[price_type][ticker]
            upper, middle, lower = talib.BBANDS(series, timeperiod=n, nbdevup=nbdevup, nbdevdn=nbdevdn, matype=matype)
            rolling_std = series.rolling(n).std()
            self.features[(ticker, f"bbands_upper_std_ratio_{n}")] = (series - upper) / rolling_std
            self.features[(ticker, f"bbands_lower_std_ratio_{n}")] = (series - lower) / rolling_std
        return self


    def add_volume_ratio(self, windows=[10,20,50]):
        for ticker in self.tickers:
            series = self.df_data_raw["Volume"][ticker]
            for window in windows:
                rolling_mean = series.rolling(window).mean()
                self.features[(ticker, f"volume_ratio_{window}")] = series / rolling_mean
        return self






# Feature Transformer

In [ ]:
class FeatureTransformer:
    def __init__(self, df_train: pd.DataFrame, df_test: pd.DataFrame):
        """
        Initialize with train and test DataFrames.

        Args:
            df_train (pd.DataFrame): Training set
            df_test (pd.DataFrame): Test set
        """
        self.df_train = df_train.copy()
        self.df_test = df_test.copy()
        self.scaler = None
        self.pca = None
        self.explained_variance_ratio_ = None
        self.train_scaled = None
        self.test_scaled = None

    # -------------------- Scale Data --------------------
    def scale(self, method: str = "standard"):
        """
        Scale train and test DataFrames using chosen method.

        Args:
            method (str): "standard" (z-score scaling) or "minmax" (0-1 scaling)
        """
        if method == "standard":
            self.scaler = StandardScaler()
        elif method == "minmax":
            self.scaler = MinMaxScaler()
        else:
            raise ValueError("method must be either 'standard' or 'minmax'")

        self.train_scaled = self.scaler.fit_transform(self.df_train)
        self.test_scaled = self.scaler.transform(self.df_test)
        return self

    # -------------------- Replace NaNs --------------------
    def fillna_zero(self):
        """
        Replace NaN values with 0 in training and test data (if they exist).
        """
        if self.train_scaled is not None:
            self.train_scaled = np.nan_to_num(self.train_scaled, nan=0.0)
        if hasattr(self, "test_scaled") and self.test_scaled is not None:
            self.test_scaled = np.nan_to_num(self.test_scaled, nan=0.0)
        return self


    # -------------------- Fit PCA --------------------
    def fit_pca(self, n_components=None):
        if self.train_scaled is None:
            raise ValueError("Data not scaled yet. Call scale() first.")

        self.pca = PCA(n_components=n_components)
        self.pca.fit(self.train_scaled)
        self.explained_variance_ratio_ = self.pca.explained_variance_ratio_
        return self




    # -------------------- Elbow Plot --------------------
    def plot_elbow(self):
        if self.explained_variance_ratio_ is None:
            raise ValueError("PCA not fitted yet. Call fit_pca() first.")

        cumulative_var = np.cumsum(self.explained_variance_ratio_)
        plt.figure(figsize=(8, 5))
        plt.plot(np.arange(1, len(cumulative_var) + 1), cumulative_var, marker='o')
        plt.xlabel("Number of Components")
        plt.ylabel("Cumulative Explained Variance")
        plt.title("PCA Elbow Plot")
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.show()

    # -------------------- Transform Data --------------------
    def transform(self, n_features=None):
        if self.pca is None:
            raise ValueError("PCA not fitted yet. Call fit_pca() first.")

        n_features = n_features or self.pca.n_components_
        train_pca = self.pca.transform(self.train_scaled)[:, :n_features]
        test_pca = self.pca.transform(self.test_scaled)[:, :n_features]

        train_df = pd.DataFrame(train_pca, index=self.df_train.index,
                                columns=[f"PCA_{i+1}" for i in range(n_features)])
        test_df = pd.DataFrame(test_pca, index=self.df_test.index,
                               columns=[f"PCA_{i+1}" for i in range(n_features)])
        return train_df, test_df

    # -------------------- Access Scaled Data --------------------
    def get_scaled_data(self):
        if self.train_scaled is None or self.test_scaled is None:
            raise ValueError("Data not scaled yet. Call scale() first.")

        train_df = pd.DataFrame(self.train_scaled, index=self.df_train.index, columns=self.df_train.columns)
        test_df = pd.DataFrame(self.test_scaled, index=self.df_test.index, columns=self.df_test.columns)
        return train_df, test_df


# Tran Test

In [ ]:
def Train_test_split(df, split_date):

    split_date = pd.to_datetime(split_date)
    df_train = df[df.index <= split_date].copy()
    df_test = df[df.index > split_date].copy()
    return df_train, df_test


def Crop_df_by_date(df, start_date=None, end_date=None):

    if start_date is not None:
        start_date = pd.to_datetime(start_date)
        df = df[df.index >= start_date]
    if end_date is not None:
        end_date = pd.to_datetime(end_date)
        df = df[df.index <= end_date]
    return df.copy()

# Financial metrics

In [ ]:
import numpy as np
import pandas as pd

def Calculate_fin_metrics(df, risk_free_rate=0.0, freq=252):
    """
    Calculate performance metrics for all log return columns in df.

    Args:
        df (pd.DataFrame): DataFrame with columns containing 'return'
        risk_free_rate (float): Annualized risk-free rate (default=0.0)
        freq (int): Trading days per year (252 for daily, 12 for monthly)

    Returns:
        pd.DataFrame: Metrics summary
    """
    results = []

    for col in df.columns:
        if "return" in col.lower():  # match return columns
            series = df[col].dropna()

            # Cumulative log returns
            cum_log = series.cumsum()

            # Wealth index
            wealth = np.exp(cum_log)

            # Total return
            total_return = cum_log.iloc[-1]

            # Annualized return (CAGR)
            ann_return = np.exp(cum_log.iloc[-1] * freq / len(series)) - 1

            # Daily volatility (log returns std)
            vol = series.std()

            # Annualized volatility
            ann_vol = vol * np.sqrt(freq)

            # Sharpe ratio
            excess_ret = series.mean() * freq - risk_free_rate
            sharpe = excess_ret / ann_vol if ann_vol != 0 else np.nan

            # Sortino ratio
            downside = series[series < 0]
            downside_vol = downside.std() * np.sqrt(freq) if not downside.empty else np.nan
            sortino = excess_ret / downside_vol if downside_vol not in [0, np.nan] else np.nan

            # Max drawdown (on wealth curve)
            roll_max = cum_log.cummax()

            # drawdown
            drawdown = roll_max - cum_log

            # max drawdown
            max_dd = drawdown.max()

            #---------------------------------------

            # CAGR = ann_return
            cagr = ann_return

            # Calmar ratio
            calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

            # Skewness & Kurtosis
            skew = series.skew()
            kurt = series.kurt()

            # VaR & CVaR
            var_95 = np.percentile(series, 5)
            cvar_95 = series[series <= var_95].mean() if not series[series <= var_95].empty else np.nan

            # Hit Ratio
            hit_ratio = (series > 0).mean()

            # Clean name (strip suffix)
            clean_name = col.replace("_returns", "")

            results.append({
                "Column": clean_name,
                "Total Return": total_return,
                "Annualized Return": ann_return,
                "Volatility": vol,
                "Annualized Volatility": ann_vol,
                "Sharpe Ratio": sharpe,
                "Sortino Ratio": sortino,
                "Calmar Ratio": calmar,
                "Max Drawdown": max_dd,
                "CAGR": cagr,
                "Skewness": skew,
                "Kurtosis": kurt,
                "VaR 95%": var_95,
                "CVaR 95%": cvar_95,
                "Hit Ratio": hit_ratio
            })

    # Build DataFrame once after loop
    res = pd.DataFrame(results).set_index("Column")
    res = res.round(3)

    return res
